# Chapter 14: Projective Quadrics

Source span: printed Volume II pages 116-145; PDF pages 125-154.

A projective quadric is the shadow cast by a homogeneous quadratic form. That sentence is short, but it carries the whole chapter: scaling a vector should not change the point it represents, so only equations homogeneous of degree two can descend cleanly to projective space. The chapter asks what survives this descent from quadratic forms: rank, signature, radical, tangency, polarity, pencils, topology, and symmetry.

The computational viewpoint in this notebook is deliberately concrete. We encode a quadric by a symmetric matrix `A` and a point of projective space by a nonzero homogeneous vector `x`, with the agreement that `x` and `lambda*x` name the same point. The image of the quadric is the set of projective points satisfying

$$x^T A x = 0.$$

Most of the geometry is then linear algebra with a projective conscience: formulas are allowed only when they do not depend on the chosen representative of a point or equation.

## Translation Guide

| Book idea | Computational object | What to inspect |
| --- | --- | --- |
| Projective space `P(E)` | Nonzero vectors modulo scalar multiplication | Use homogeneous coordinates and dehomogenize only after choosing a chart. |
| Space of quadrics `PQ(E)` | Nonzero symmetric matrices modulo scalar multiplication | Multiplying `A` by a nonzero scalar leaves the same quadric. |
| Image of a quadric | Zero set of `x.T @ A @ x` in projective space | Rank and signature explain whether the chart looks empty, singular, oval-like, or ruled. |
| Pencil of quadrics | A projective line `A0 + t A1` in the matrix space | The determinant along the pencil finds degenerate members. |
| Tangent hyperplane at `[x]` | The linear equation `(A @ x).T y = 0` | The gradient of the quadratic equation is the polar hyperplane. |
| Polarity | The map `[x] -> [A @ x]` when `A` is invertible | Points become hyperplanes; incidence becomes conjugacy. |
| Tangential quadric | The quadric in the dual projective space with matrix `A^{-1}` | A hyperplane is tangent exactly when its dual coordinates satisfy the dual equation. |
| Projective group of a proper quadric | Matrices preserving `A` up to scalar | Transform points and transform the equation contragrediently. |

The safest mental model is this: a projective quadric is not just a surface drawn in an affine chart. It is an equivalence class of equations, and its chart picture may split, disappear, or acquire points at infinity depending on where the chart cuts it.

In [ ]:
from pathlib import Path
import sys
import math

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go


def locate_book_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "utils").is_dir():
            return candidate
    raise RuntimeError("Could not locate the Geometry II book root")


BOOK_ROOT = locate_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, assert_artifacts, display_artifact, save_csv, save_json, save_plotly_html
from utils.geometry import homogeneous, polar_line, quadratic_signature
from utils.plotting import COLORS, new_axes, save_figure

CHAPTER = 14
ARTIFACTS: list[Path] = []


def chapter_artifact(kind: str, filename: str) -> Path:
    return artifact_path(CHAPTER, kind, filename, book_root=BOOK_ROOT)


print(f"BOOK_ROOT = {BOOK_ROOT}")

## Visualization Storyboard

We will build the chapter around five visual questions.

1. What does rank and signature do to the real chart picture of a projective conic?
2. How does a pencil of quadrics become a line in the projective space of equations, and where do its degenerate members sit?
3. Why are some projective quadrics ruled by lines, even when an affine drawing looks curved?
4. How does polarity turn a point into a tangent or secant construction?
5. What is the dual, or tangential, equation of a proper quadric?

Each picture is paired with a numeric check. The checks are not decoration: they are the guardrails that keep the affine drawings honest about the projective objects behind them.

In [ ]:
def sym_outer(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return 0.5 * (np.outer(a, b) + np.outer(b, a))


def line_through(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    line = np.cross(np.asarray(p, dtype=float), np.asarray(q, dtype=float))
    return line / np.linalg.norm(line[:2])


def conic_from_lines(l1: np.ndarray, l2: np.ndarray) -> np.ndarray:
    return sym_outer(l1, l2)


def q_value(A: np.ndarray, x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return float(x @ A @ x)


def q_values(A: np.ndarray, points_h: np.ndarray) -> np.ndarray:
    return np.einsum("...i,ij,...j->...", points_h, A, points_h)


def affine_conic_values(A: np.ndarray, X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    return (
        A[0, 0] * X * X
        + 2 * A[0, 1] * X * Y
        + 2 * A[0, 2] * X
        + A[1, 1] * Y * Y
        + 2 * A[1, 2] * Y
        + A[2, 2]
    )


def draw_affine_line(ax, line: np.ndarray, *, xlim=(-2.5, 2.5), color=COLORS["gray"], label=None, linewidth=1.7):
    a, b, c = line
    xs = np.linspace(*xlim, 240)
    if abs(b) > 1e-12:
        ys = -(a * xs + c) / b
        ax.plot(xs, ys, color=color, linewidth=linewidth, label=label)
    elif abs(a) > 1e-12:
        ax.axvline(-c / a, color=color, linewidth=linewidth, label=label)


def plot_zero_contour(ax, A: np.ndarray, *, color=COLORS["blue"], linewidth=2.2, label=None, span=2.3):
    grid = np.linspace(-span, span, 401)
    X, Y = np.meshgrid(grid, grid)
    Z = affine_conic_values(A, X, Y)
    if np.nanmin(Z) <= 0 <= np.nanmax(Z):
        contour = ax.contour(X, Y, Z, levels=[0], colors=[color], linewidths=linewidth)
        if label:
            try:
                contour.collections[0].set_label(label)
            except AttributeError:
                contour.set_label(label)
        return contour
    return None


def rank_and_signature(A: np.ndarray) -> tuple[int, tuple[int, int, int]]:
    return int(np.linalg.matrix_rank(A, tol=1e-9)), quadratic_signature(A)


print("Helper functions ready")

## 1. Equations, Images, Rank, And Signature

A quadric in `P(E)` is an equation class rather than one preferred equation. In coordinates, replacing `A` by `c*A` does not change the zero set, and replacing `x` by `lambda*x` multiplies the residual by `lambda^2`. This is why the zero condition is projectively meaningful.

Over the real numbers, the visible chart can be misleading. A proper conic can look like an oval in one chart and like two hyperbola branches in another. A degenerate equation can be a point, a repeated line, or two lines. The matrix rank detects degeneracy, while the signature records the positive and negative directions of the form. The following gallery uses the affine chart `z = 1`, so it is a chart-level view of projective data, not a complete taxonomy by itself.

In [ ]:
gallery = [
    ("empty real image", np.diag([1.0, 1.0, 1.0]), "rank 3, all signs equal"),
    ("single point", np.diag([1.0, 1.0, 0.0]), "rank 2, semidefinite"),
    ("double line", np.diag([1.0, 0.0, 0.0]), "rank 1"),
    ("two lines", np.diag([1.0, -1.0, 0.0]), "rank 2, indefinite"),
    ("oval chart", np.diag([1.0, 1.0, -1.0]), "proper, one negative"),
    ("two affine arcs", np.diag([1.0, -1.0, -1.0]), "proper, line at infinity cuts it"),
]

fig, axes = plt.subplots(2, 3, figsize=(11.5, 7.2), constrained_layout=True)
for ax, (title, A, note) in zip(axes.flat, gallery):
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-2.35, 2.35)
    ax.set_ylim(-2.35, 2.35)
    ax.grid(True, color="#e5e7eb", linewidth=0.8)
    ax.axhline(0, color="#d1d5db", linewidth=0.9)
    ax.axvline(0, color="#d1d5db", linewidth=0.9)
    rank, sig = rank_and_signature(A)
    ax.set_title(f"{title}\nrank {rank}, signature {sig}", fontsize=10, loc="left")
    if title == "empty real image":
        ax.text(0, 0, "no real chart\npoints", ha="center", va="center", color=COLORS["gray"], fontsize=10)
    elif title == "single point":
        ax.scatter([0], [0], s=62, color=COLORS["orange"], edgecolor="white", zorder=5)
        ax.text(0.14, 0.12, "one projective point", fontsize=8, color=COLORS["gray"])
    elif title == "double line":
        ax.axvline(0, color=COLORS["purple"], linewidth=4.2, alpha=0.55)
        ax.text(0.14, 1.65, "x = 0 twice", fontsize=8, color=COLORS["gray"])
    elif title == "two lines":
        ax.plot([-2.3, 2.3], [-2.3, 2.3], color=COLORS["teal"], linewidth=2.2)
        ax.plot([-2.3, 2.3], [2.3, -2.3], color=COLORS["teal"], linewidth=2.2)
    else:
        plot_zero_contour(ax, A, color=COLORS["blue"], linewidth=2.4, span=2.35)
    ax.text(-2.18, -2.13, note, fontsize=8, color=COLORS["gray"])
    ax.tick_params(labelsize=8)

signature_gallery = chapter_artifact("figures", "rank_signature_gallery.png")
save_figure(fig, signature_gallery)
ARTIFACTS.append(signature_gallery)
display_artifact(signature_gallery, width=920)

## 2. Pencils: Lines In The Space Of Equations

The vector space of quadratic forms has a projectivization of its own. A pencil is a projective line in that space, so two equations `A0` and `A1` generate members `A0 + t A1`. A familiar plane-conic construction appears when we choose four base points and take products of opposite line pairs. Every member of the pencil passes through the same four points, while the determinant along the pencil marks the degenerate conics.

The important projective habit is to separate two questions. The base locus asks which points satisfy every equation in the pencil. The discriminant asks which equations in the pencil fail to be proper. They live in different projective spaces, but the computation ties them together through one cubic determinant.

In [ ]:
base_affine = np.array([
    [-1.25, -0.72],
    [1.18, -0.52],
    [0.95, 1.08],
    [-1.12, 0.86],
])
base_h = homogeneous(base_affine)

L12 = line_through(base_h[0], base_h[1])
L34 = line_through(base_h[2], base_h[3])
L13 = line_through(base_h[0], base_h[2])
L24 = line_through(base_h[1], base_h[3])
A0 = conic_from_lines(L12, L34)
A1 = conic_from_lines(L13, L24)

ts_to_draw = [-1.8, -0.85, -0.25, 0.55, 1.35]
colors = [COLORS["purple"], COLORS["blue"], COLORS["green"], COLORS["orange"], COLORS["red"]]

fig, ax = new_axes(figsize=(7.6, 6.2), title="A conic pencil through four fixed base points")
ax.set_xlim(-2.25, 2.25)
ax.set_ylim(-1.85, 1.85)
for t, color in zip(ts_to_draw, colors):
    plot_zero_contour(ax, A0 + t * A1, color=color, linewidth=1.9, label=f"t={t:g}", span=2.4)
ax.scatter(base_affine[:, 0], base_affine[:, 1], s=54, color=COLORS["ink"], edgecolor="white", zorder=6, label="base points")
for i, point in enumerate(base_affine, start=1):
    ax.text(point[0] + 0.05, point[1] + 0.05, f"m{i}", fontsize=9)
ax.legend(loc="upper right", fontsize=8, frameon=True)
pencil_png = chapter_artifact("figures", "pencil_four_base_conics.png")
save_figure(fig, pencil_png)
ARTIFACTS.append(pencil_png)

ts = np.linspace(-4.0, 4.0, 401)
dets = np.array([np.linalg.det(A0 + t * A1) for t in ts])
coeff = np.polyfit(ts, dets, 3)
roots = np.roots(coeff)
real_roots = sorted(float(root.real) for root in roots if abs(root.imag) < 1e-7 and ts[0] <= root.real <= ts[-1])
root_rows = [
    {"root_t": root, "determinant": float(np.linalg.det(A0 + root * A1))}
    for root in real_roots
]
roots_csv = chapter_artifact("tables", "pencil_degenerate_roots.csv")
save_csv(root_rows, roots_csv)
ARTIFACTS.append(roots_csv)

fig_det = go.Figure()
fig_det.add_trace(go.Scatter(x=ts, y=dets, mode="lines", name="det(A0 + t A1)", line=dict(color="#2f6fed", width=3)))
fig_det.add_trace(go.Scatter(x=[ts[0], ts[-1]], y=[0, 0], mode="lines", name="zero", line=dict(color="#6b7280", dash="dash")))
if real_roots:
    fig_det.add_trace(go.Scatter(
        x=real_roots,
        y=[0] * len(real_roots),
        mode="markers+text",
        text=[f"t={root:.3g}" for root in real_roots],
        textposition="top center",
        marker=dict(size=10, color="#d97706"),
        name="degenerate members",
    ))
fig_det.update_layout(
    title="Discriminant slice of the pencil",
    xaxis_title="pencil parameter t",
    yaxis_title="determinant",
    template="plotly_white",
    height=460,
)
pencil_html = chapter_artifact("plots", "pencil_discriminant.html")
save_plotly_html(fig_det, pencil_html)
ARTIFACTS.append(pencil_html)

display_artifact(pencil_png, width=760)
display_artifact(pencil_html, width=780, height=500)

## 3. Real Topology And Ruled Quadrics

For a proper real quadric, the signature controls more than the local drawing. It controls the topology after projectivization. In low dimension the nonempty proper conics are circles, but in projective three-space a neutral quadric has two systems of generating lines. In an affine chart this can show up as a hyperbolic paraboloid or a one-sheeted hyperboloid.

The next visual uses the affine equation `z = x*y`, which is the chart `w = 1` of the homogeneous equation

$$x y - z w = 0.$$

Holding `y` fixed gives one straight-line ruling; holding `x` fixed gives the other. The surface bends, but each ruling is made from honest projective lines lying entirely on the quadric.

In [ ]:
u = np.linspace(-1.8, 1.8, 75)
v = np.linspace(-1.8, 1.8, 75)
U, V = np.meshgrid(u, v)
Z = U * V

fig_ruled = go.Figure()
fig_ruled.add_trace(go.Surface(
    x=U,
    y=V,
    z=Z,
    colorscale=[[0, "#dbeafe"], [1, "#93c5fd"]],
    opacity=0.68,
    showscale=False,
    name="xy - zw = 0",
))
for v0 in [-1.4, -0.7, 0.0, 0.7, 1.4]:
    xs = u
    ys = np.full_like(u, v0)
    fig_ruled.add_trace(go.Scatter3d(x=xs, y=ys, z=xs * ys, mode="lines", line=dict(color="#0f9d8f", width=6), name="y fixed"))
for u0 in [-1.4, -0.7, 0.0, 0.7, 1.4]:
    ys = v
    xs = np.full_like(v, u0)
    fig_ruled.add_trace(go.Scatter3d(x=xs, y=ys, z=xs * ys, mode="lines", line=dict(color="#d97706", width=6), name="x fixed"))
fig_ruled.update_layout(
    title="Two ruling systems on an affine chart of a projective quadric",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"),
    template="plotly_white",
    height=650,
    showlegend=False,
)
ruled_html = chapter_artifact("plots", "ruled_quadric_two_families.html")
save_plotly_html(fig_ruled, ruled_html)
ARTIFACTS.append(ruled_html)

display_artifact(ruled_html, width=820, height=690)

## 4. Polarity: Points Become Hyperplanes

When the matrix `A` is invertible, the bilinear form behind the quadric gives an isomorphism from points to hyperplanes. For a representative `x`, the polar hyperplane has equation

$$(A x)^T y = 0.$$

If `x` lies on the quadric, this hyperplane is tangent at `x`. If `x` is outside a real conic, the polar line joins the two contact points of the tangents from `x`. If `x` is inside, the same linear equation still exists, but the contact points are no longer real in that affine chart. The algebra is uniform; the real picture changes.

In [ ]:
A_circle = np.diag([1.0, 1.0, -1.0])
external = np.array([1.8, 0.65, 1.0])
external_xy = external[:2] / external[2]
polar = polar_line(A_circle, external)

distance = np.linalg.norm(external_xy)
unit = external_xy / distance
perp = np.array([-unit[1], unit[0]])
center_on_contact_chord = unit / distance
half_length = math.sqrt(1.0 - 1.0 / distance**2)
tangent_points = np.vstack([
    center_on_contact_chord + half_length * perp,
    center_on_contact_chord - half_length * perp,
])

fig, ax = new_axes(figsize=(7.2, 6.2), title="Polarity with respect to the unit conic")
ax.set_xlim(-1.6, 2.15)
ax.set_ylim(-1.45, 1.45)
theta = np.linspace(0, 2 * np.pi, 500)
ax.plot(np.cos(theta), np.sin(theta), color=COLORS["ink"], linewidth=2.2, label="quadric")
draw_affine_line(ax, polar, xlim=(-1.6, 2.15), color=COLORS["blue"], label="polar line", linewidth=2.1)
for pt in tangent_points:
    ax.plot([external_xy[0], pt[0]], [external_xy[1], pt[1]], color=COLORS["orange"], linewidth=1.8)
ax.scatter([external_xy[0]], [external_xy[1]], color=COLORS["red"], s=58, edgecolor="white", zorder=6, label="point m")
ax.scatter(tangent_points[:, 0], tangent_points[:, 1], color=COLORS["teal"], s=52, edgecolor="white", zorder=6, label="contact points")
ax.text(external_xy[0] + 0.04, external_xy[1] + 0.06, "m", fontsize=10)
ax.legend(loc="lower left", fontsize=8)
polarity_png = chapter_artifact("figures", "polarity_tangent_contacts.png")
save_figure(fig, polarity_png)
ARTIFACTS.append(polarity_png)

display_artifact(polarity_png, width=720)

## 5. Tangential Quadrics: The Dual Equation

A tangent line to a conic can be described as a point of the dual projective plane. If the original proper conic has matrix `A`, then the dual conic of tangent hyperplanes has matrix `A^{-1}`. In other words, a hyperplane with coefficient vector `ell` is tangent exactly when

$$ell^T A^{-1} ell = 0.$$

For the unit conic `x^2 + y^2 - z^2 = 0`, the tangent at angle `theta` has coefficients `(cos(theta), sin(theta), -1)`. These coefficients trace another conic in the dual chart. The left panel below shows the tangent envelope; the right panel shows the same family as points satisfying the tangential equation.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 28, endpoint=False)
dual_lines = np.column_stack([np.cos(angles), np.sin(angles), -np.ones_like(angles)])
A_dual = np.linalg.inv(A_circle)

theta_dense = np.linspace(0, 2 * np.pi, 500)
fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.2), constrained_layout=True)
left, right = axes
for ax in axes:
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#e5e7eb", linewidth=0.8)
    ax.tick_params(labelsize=8)

left.set_title("tangent lines in P(E)", loc="left", fontsize=11)
left.plot(np.cos(theta_dense), np.sin(theta_dense), color=COLORS["ink"], linewidth=2.0)
for ell in dual_lines[::2]:
    draw_affine_line(left, ell, xlim=(-1.6, 1.6), color=COLORS["blue"], linewidth=1.0)
left.set_xlim(-1.55, 1.55)
left.set_ylim(-1.35, 1.35)

right.set_title("the same family in the dual chart", loc="left", fontsize=11)
right.plot(np.cos(theta_dense), np.sin(theta_dense), color=COLORS["orange"], linewidth=2.0, label="dual conic")
right.scatter(dual_lines[:, 0], dual_lines[:, 1], s=24, color=COLORS["blue"], edgecolor="white", zorder=5)
right.set_xlim(-1.25, 1.25)
right.set_ylim(-1.25, 1.25)
right.set_xlabel("xi / -zeta")
right.set_ylabel("eta / -zeta")

tangential_png = chapter_artifact("figures", "tangential_dual_conic.png")
save_figure(fig, tangential_png)
ARTIFACTS.append(tangential_png)

display_artifact(tangential_png, width=900)

## 6. Projective Changes Of Coordinates

A projective transformation moves points and changes equations in opposite directions. If `y = H x`, then the original equation `x.T @ A @ x = 0` becomes

$$y^T H^{-T} A H^{-1} y = 0.$$

This contragredient rule is the computational form of the projective group action on quadrics. It is also the reason polarity is natural rather than accidental: points and hyperplanes transform by dual rules.

In [ ]:
H = np.array([
    [1.05, 0.35, 0.15],
    [-0.20, 0.95, -0.05],
    [0.22, -0.12, 1.00],
])
H_inv = np.linalg.inv(H)
A_transformed = H_inv.T @ A_circle @ H_inv
circle_h = np.column_stack([np.cos(theta_dense), np.sin(theta_dense), np.ones_like(theta_dense)])
transformed_h = (H @ circle_h.T).T
transformed_xy = transformed_h[:, :2] / transformed_h[:, 2:3]

fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.2), constrained_layout=True)
for ax in axes:
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#e5e7eb", linewidth=0.8)
    ax.tick_params(labelsize=8)
axes[0].set_title("original equation", loc="left", fontsize=11)
axes[0].plot(circle_h[:, 0], circle_h[:, 1], color=COLORS["ink"], linewidth=2.1)
axes[0].set_xlim(-1.35, 1.35)
axes[0].set_ylim(-1.35, 1.35)
axes[1].set_title("transformed equation", loc="left", fontsize=11)
plot_zero_contour(axes[1], A_transformed, color=COLORS["blue"], linewidth=2.2, span=2.0)
axes[1].scatter(transformed_xy[::22, 0], transformed_xy[::22, 1], s=18, color=COLORS["orange"], zorder=5, label="transformed sample points")
axes[1].legend(loc="lower left", fontsize=8)
axes[1].set_xlim(-1.65, 1.85)
axes[1].set_ylim(-1.55, 1.55)

projective_action_png = chapter_artifact("figures", "projective_action_on_conic.png")
save_figure(fig, projective_action_png)
ARTIFACTS.append(projective_action_png)

display_artifact(projective_action_png, width=900)

## Applied Lab: Design A Quadric By Constraints

Here is a compact workflow for designing with projective quadrics.

1. Choose the ambient projective dimension and represent an unknown symmetric matrix `A`.
2. Convert incidence constraints into linear equations in the entries of `A`: a point `[x]` lies on the quadric when `x.T @ A @ x = 0`.
3. Convert tangency or polarity constraints into equations involving `A @ x`.
4. If the result is a pencil, study the determinant along the pencil before choosing a member.
5. Only after the projective object is selected should you choose an affine chart for drawing or fabrication.

This order prevents a common mistake: designing from the chart picture alone. Projective quadrics remember points at infinity, dual hyperplanes, and degenerate limits that a single chart can hide.

In [ ]:
def final_sanity() -> dict[str, float | int | list[str]]:
    scale_A = np.array([
        [2.0, 0.3, -0.2],
        [0.3, -1.1, 0.4],
        [-0.2, 0.4, 0.7],
    ])
    x = np.array([0.8, -1.4, 1.0])
    lam = -3.25
    scale_error = abs(q_value(scale_A, lam * x) - lam**2 * q_value(scale_A, x))

    transformed_residuals = q_values(A_transformed, transformed_h)
    original_residuals = q_values(A_circle, circle_h)
    action_error = float(np.max(np.abs(transformed_residuals - original_residuals)))

    base_residual = float(max(
        np.max(np.abs(q_values(A0 + t * A1, base_h)))
        for t in [-2.0, -0.3, 0.0, 0.9, 2.2]
    ))
    root_det_error = float(max([abs(np.linalg.det(A0 + root * A1)) for root in real_roots] or [0.0]))

    tangent_h = homogeneous(tangent_points)
    contact_on_conic_error = float(np.max(np.abs(q_values(A_circle, tangent_h))))
    contact_on_polar_error = float(np.max(np.abs(tangent_h @ polar)))

    dual_errors = np.abs(q_values(A_dual, dual_lines))
    dual_error = float(np.max(dual_errors))

    A_ruled = np.zeros((4, 4))
    A_ruled[0, 1] = A_ruled[1, 0] = 0.5
    A_ruled[2, 3] = A_ruled[3, 2] = -0.5
    sample_u = np.linspace(-1.7, 1.7, 31)
    sample_v = np.linspace(-1.5, 1.5, 29)
    ruled_points = np.array([[a, b, a * b, 1.0] for a in sample_u for b in sample_v])
    ruled_error = float(np.max(np.abs(q_values(A_ruled, ruled_points))))

    checks = {
        "scale_error": scale_error,
        "projective_action_error": action_error,
        "pencil_base_residual": base_residual,
        "pencil_root_determinant_error": root_det_error,
        "polarity_contact_on_conic_error": contact_on_conic_error,
        "polarity_contact_on_polar_error": contact_on_polar_error,
        "tangential_dual_error": dual_error,
        "ruled_quadric_residual": ruled_error,
        "artifact_count": len(ARTIFACTS) + 1,
        "artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in ARTIFACTS],
    }

    checks_path = chapter_artifact("checks", "projective_quadric_checks.json")
    save_json(checks, checks_path)
    if checks_path not in ARTIFACTS:
        ARTIFACTS.append(checks_path)

    assert scale_error < 1e-10
    assert action_error < 1e-10
    assert base_residual < 1e-10
    assert root_det_error < 1e-8
    assert contact_on_conic_error < 1e-10
    assert contact_on_polar_error < 1e-10
    assert dual_error < 1e-10
    assert ruled_error < 1e-10
    assert_artifacts(ARTIFACTS, min_bytes=64)
    return checks


checks = final_sanity()
checks

## Takeaways

A projective quadric is best learned as a matrix class plus a zero set. The matrix class remembers which operations are legal under scaling; the zero set tells us what the geometry looks like in projective space.

Rank and signature give the first classification language. Pencils add a second layer: the space of equations has its own projective geometry, and a line in that space meets the discriminant where the member becomes degenerate. Polarity is the chapter's central duality mechanism. For a proper quadric, multiplying by `A` turns points into hyperplanes, tangency into a single linear equation, and tangent hyperplanes into a dual quadric with matrix `A^{-1}`.

The pictures in an affine chart are useful, but they are not the object itself. The same projective quadric may look like an oval, two arcs, or a ruled surface depending on the chart and dimension. The checks above keep the invariant statements visible underneath those drawings.